<a href="https://colab.research.google.com/github/SebGalvaleira/mineria-datos-ii-parcial/blob/main/notebooks/e2/E2_05_pruebas_controladas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2 · 05 · Pruebas controladas de punta a punta
Sandbox con el mismo pipeline (stream → Bronze → Silver → Gold) para verificar, con casos inyectados, el comportamiento
ante (1) eventos duplicados, (2) un evento extremadamente tardío, (3) un micro-lote reprocesado después de un fallo
y (4) un duplicado que llega hasta Silver. Cada prueba compara Gold antes y después y termina con una verificación automática.

In [2]:
!pip install -q "pyspark>=4.0,<4.1"
import os, shutil, subprocess
os.environ.pop('JAVA_HOME', None)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
LAKE_DRIVE = Path('/content/drive/MyDrive/cpa_datalake')   # respaldo persistente
LAKE       = Path('/content/cpa_datalake')                  # copia de trabajo (disco local de Colab)

RUTAS = {z: LAKE / z for z in ['landing', 'stream_input', 'bronze', 'silver',
                               'gold', 'quarantine', 'checkpoints', 'logs']}

def restaurar(*rutas):
    """Copia desde Drive SOLO las carpetas que este notebook necesita (si no están ya en local)."""
    for r in rutas:
        origen, destino = LAKE_DRIVE / r, LAKE / r
        if origen.exists() and not destino.exists():
            shutil.copytree(origen, destino)
            print('✓ restaurado:', r)

def guardar_en_drive(*rutas):
    """Sincroniza en Drive SOLO las carpetas indicadas."""
    for r in rutas:
        origen, destino = LAKE / r, LAKE_DRIVE / r
        destino.mkdir(parents=True, exist_ok=True)
        subprocess.run(['rsync', '-a', '--delete', f'{origen}/', f'{destino}/'], check=True)
        print('✓ sincronizado:', r)

from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder.appName('cloud-provider-analytics')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')
         .getOrCreate())
print('Spark', spark.version)

Mounted at /content/drive
Spark 4.0.5


In [3]:
#restaura lo necesario y prepara el sandbox
import json
restaurar('landing', 'bronze/customers_orgs', 'bronze/resources', 'logs')
for p in RUTAS.values():
    p.mkdir(parents=True, exist_ok=True)

SB = LAKE / 'pruebas'
shutil.rmtree(SB, ignore_errors=True)                    # el laboratorio arranca vacío
P = {k: SB / k for k in ['entrada', 'bronze', 'checkpoint', 'silver', 'gold', 'quarantine']}
for p in P.values():
    p.mkdir(parents=True, exist_ok=True)

UMBRALES = json.load(open(RUTAS['logs'] / 'umbrales.json'))      # los mismos del notebook 03
HISTORIA = sorted((RUTAS['landing'] / 'usage_events_stream').glob('*.jsonl'))[:10]
print('Umbrales:', UMBRALES, '· archivos de historia:', len(HISTORIA))

✓ restaurado: landing
✓ restaurado: bronze/customers_orgs
✓ restaurado: bronze/resources
✓ restaurado: logs
Umbrales: {'p99_costo': 16.7175, 'pico_costo': 83.5875, 'costo_minimo_valido': -0.01} · archivos de historia: 10


In [4]:
#dimensiones y esquemas (iguales a 02 y 03)
def ultimo_snapshot(df):
    return df.filter(F.col('ingest_date') == df.agg(F.max('ingest_date')).first()[0])

orgs = ultimo_snapshot(spark.read.parquet(str(RUTAS['bronze'] / 'customers_orgs')))
res  = ultimo_snapshot(spark.read.parquet(str(RUTAS['bronze'] / 'resources')))
ORGS_DIM = F.broadcast(orgs.select('org_id', 'org_name', F.lower(F.trim('industry')).alias('industry'),
                                   F.lower(F.trim('plan_tier')).alias('plan_tier'), 'hq_region', 'is_enterprise'))
RES_DIM  = F.broadcast(res.select('resource_id', F.col('state').alias('resource_state')))

S, D = T.StringType(), T.DoubleType()
campo = lambda n, t: T.StructField(n, t, True)
ESQUEMA_EVENTOS = T.StructType([
    campo('event_id', S), campo('timestamp', T.TimestampType()), campo('org_id', S), campo('resource_id', S),
    campo('service', S), campo('region', S), campo('metric', S), campo('value', S), campo('unit', S),
    campo('cost_usd_increment', D), campo('schema_version', T.IntegerType()), campo('carbon_kg', D),
    campo('genai_tokens', T.LongType()), campo('_corrupt_record', S)])
MAPA_UNIDAD = F.create_map(*[F.lit(x) for x in ['requests', 'count', 'cpu_hours', 'hours',
                                                'storage_gb_hours', 'gb_hours']])

In [5]:
#pipeline del sandbox
#son las mismas funciones del 02, 03 y 04. juntas. (Que esten repetidas es una deuda tecnica)
FALLA, AFECTADAS, LOG = {'activa': False}, set(), []

# --- Streaming → Bronze (igual al 02, con la corrección de fechas afectadas) ---
def procesar_lote(df, batch_id):
    df = df.persist()
    validos = (df.filter(F.col('_corrupt_record').isNull() & F.col('event_id').isNotNull())
                 .drop('_corrupt_record', 'dedupe_key'))
    if any(P['bronze'].glob('event_date=*')):
        nuevos = validos.join(spark.read.parquet(str(P['bronze'])).select('event_id'), 'event_id', 'left_anti')
    else:
        nuevos = validos
    nuevos = nuevos.withColumn('stream_batch_id', F.lit(batch_id)).persist()
    fechas = sorted(str(r[0]) for r in validos.select('event_date').distinct().collect())   # ← TODOS los válidos
    n = {'batch_id': batch_id, 'entrada': df.count(), 'validos': validos.count(),
         'nuevos': nuevos.count(), 'fechas': fechas}
    n['ya_existian'] = n['validos'] - n['nuevos']
    nuevos.write.mode('append').partitionBy('event_date').parquet(str(P['bronze']))
    if FALLA['activa']:
        FALLA['activa'] = False
        raise RuntimeError(f'Falla simulada en el lote {batch_id}')
    AFECTADAS.update(fechas)                  # solo si el lote terminó bien
    LOG.append(n); print(' lote:', n)
    nuevos.unpersist(); df.unpersist()

def correr_stream():
    stream = (spark.readStream.schema(ESQUEMA_EVENTOS).option('mode', 'PERMISSIVE')
              .option('columnNameOfCorruptRecord', '_corrupt_record').option('maxFilesPerTrigger', 10)
              .json(str(P['entrada']))
              .withColumn('ingest_ts', F.current_timestamp())
              .withColumn('event_date', F.to_date('timestamp'))
              .withColumn('dedupe_key', F.coalesce('event_id', F.expr('uuid()')))
              .withWatermark('ingest_ts', '1 hour')
              .dropDuplicatesWithinWatermark(['dedupe_key']))
    q = (stream.writeStream.foreachBatch(procesar_lote)
         .option('checkpointLocation', str(P['checkpoint'])).trigger(availableNow=True).start())
    try:
        q.awaitTermination(); return True
    except Exception as e:
        print('  stream detenido:', str(e).splitlines()[0][:150]); return False

# --- Silver (igual al 03) ---
def construir_silver(ev):
    sin_id = ev.filter(F.col('event_id').isNull())
    ev = ev.filter(F.col('event_id').isNotNull()).dropDuplicates(['event_id']).unionByName(sin_id)
    ev = (ev.withColumn('value_num', F.expr('try_cast(value as double)'))
            .withColumn('unit', F.coalesce('unit', MAPA_UNIDAD[F.col('metric')]))
            .withColumn('requests',         F.when(F.col('metric') == 'requests', F.col('value_num')))
            .withColumn('cpu_hours',        F.when(F.col('metric') == 'cpu_hours', F.col('value_num')))
            .withColumn('storage_gb_hours', F.when(F.col('metric') == 'storage_gb_hours', F.col('value_num')))
            .withColumn('flag_costo_anomalo', F.col('cost_usd_increment') < UMBRALES['costo_minimo_valido'])
            .withColumn('flag_pico_costo',    F.col('cost_usd_increment') > UMBRALES['pico_costo'])
            .join(ORGS_DIM, 'org_id', 'left').join(RES_DIM, 'resource_id', 'left'))
    reglas = [('event_id_nulo', F.col('event_id').isNull()),
              ('value_no_numerico', F.col('value').isNotNull() & F.col('value_num').isNull()),
              ('unit_nula_con_value', F.col('value').isNotNull() & F.col('unit').isNull()),
              ('schema_version_invalida', F.col('schema_version').isNull() | ~F.col('schema_version').isin(1, 2)),
              ('org_desconocida', F.col('org_name').isNull())]
    ev = ev.withColumn('motivo', F.coalesce(*[F.when(c, F.lit(n)) for n, c in reglas]))
    return ev.filter(F.col('motivo').isNull()).drop('motivo'), ev.filter(F.col('motivo').isNotNull())

def actualizar_silver(fechas):
    b = spark.read.parquet(str(P['bronze'])).filter(F.col('event_date').isin(fechas))
    silver, quarantine = construir_silver(b)
    silver.write.mode('overwrite').partitionBy('event_date', 'service').parquet(str(P['silver']))
    if quarantine.count():
        quarantine.write.mode('append').parquet(str(P['quarantine']))

# --- Gold (igual al 04) ---
def recalcular_gold(fechas):
    s = (spark.read.parquet(str(P['silver'])).filter(F.col('event_date').isin(fechas))
              .dropDuplicates(['event_id']))                              # última barrera
    mart = (s.groupBy('org_id', 'event_date', 'service')
              .agg(F.round(F.sum('cost_usd_increment'), 4).alias('daily_cost_usd'),
                   F.count('*').alias('eventos'))
              .withColumn('updated_at', F.current_timestamp()))
    mart.repartition('event_date').write.mode('overwrite').partitionBy('event_date').parquet(str(P['gold']))

# --- El pipeline completo ---
def ejecutar_pipeline():
    AFECTADAS.clear()
    ok = correr_stream()
    if not ok:
        print('  el job se cayó: Silver y Gold NO se actualizan en esta corrida'); return []
    fechas = sorted(AFECTADAS)
    if fechas:
        actualizar_silver(fechas); recalcular_gold(fechas)
    print(' fechas recalculadas:', fechas if len(fechas) <= 5 else f'{len(fechas)} fechas')
    return fechas

In [6]:
#herramientas para compara gold antes y despues

import pandas as pd

def foto_gold():
    return spark.read.parquet(str(P['gold'])).toPandas()

def diferencias(antes, despues):
    m = antes.merge(despues, on=['org_id', 'event_date', 'service'], how='outer',
                    suffixes=('_antes', '_despues'), indicator=True)
    cambio = ((m['_merge'] != 'both') |
              (m['daily_cost_usd_antes'].round(4) != m['daily_cost_usd_despues'].round(4)) |
              (m['eventos_antes'] != m['eventos_despues']))
    return m.loc[cambio, ['org_id', 'event_date', 'service', 'daily_cost_usd_antes',
                          'daily_cost_usd_despues', 'eventos_antes', 'eventos_despues']]

def contar_id(event_id, zona):
    return spark.read.parquet(str(P[zona])).filter(F.col('event_id') == event_id).count()

def escribir_archivo(nombre, eventos):
    with open(P['entrada'] / nombre, 'w') as f:
        f.write('\n'.join(json.dumps(e) for e in eventos) + '\n')

RESULTADOS = []

In [7]:
# paso 0, linea base
# 10 archivos son 3600 eventos, como dcada archivo cubre los 60 dias, la linea base ya tiene gold en todas las fechas

for h in HISTORIA:
    shutil.copy(h, P['entrada'] / h.name)
ejecutar_pipeline()
base_gold = foto_gold()
print('Bronze:', spark.read.parquet(str(P['bronze'])).count(), '· Gold:', len(base_gold), 'filas ·',
      round(base_gold.daily_cost_usd.sum(), 4), 'USD')

lineas = [json.loads(l) for l in open(HISTORIA[0])]          # eventos reales para armar los casos

 lote: {'batch_id': 0, 'entrada': 3600, 'validos': 3600, 'nuevos': 3600, 'fechas': ['2025-07-03', '2025-07-04', '2025-07-05', '2025-07-06', '2025-07-07', '2025-07-08', '2025-07-09', '2025-07-10', '2025-07-11', '2025-07-12', '2025-07-13', '2025-07-14', '2025-07-15', '2025-07-16', '2025-07-17', '2025-07-18', '2025-07-19', '2025-07-20', '2025-07-21', '2025-07-22', '2025-07-23', '2025-07-24', '2025-07-25', '2025-07-26', '2025-07-27', '2025-07-28', '2025-07-29', '2025-07-30', '2025-07-31', '2025-08-01', '2025-08-02', '2025-08-03', '2025-08-04', '2025-08-05', '2025-08-06', '2025-08-07', '2025-08-08', '2025-08-09', '2025-08-10', '2025-08-11', '2025-08-12', '2025-08-13', '2025-08-14', '2025-08-15', '2025-08-16', '2025-08-17', '2025-08-18', '2025-08-19', '2025-08-20', '2025-08-21', '2025-08-22', '2025-08-23', '2025-08-24', '2025-08-25', '2025-08-26', '2025-08-27', '2025-08-28', '2025-08-29', '2025-08-30', '2025-08-31'], 'ya_existian': 0}
 lote: {'batch_id': 1, 'entrada': 0, 'validos': 0, 'nuevo

In [8]:
# prueba 1, pruebas duplicadas:
# el archivo trae 5 lineas, pero en gold solo debe aparecer el evento nuevo, una vez
antes = foto_gold()
reenviados = lineas[:3]                                                   # 3 eventos que YA están en Bronze
nuevo = {**lineas[3], 'event_id': 'evt_test_dup', 'cost_usd_increment': 1.0}
escribir_archivo('t1_duplicados.jsonl', reenviados + [nuevo, nuevo])      # 5 líneas: 3 reenvíos + 1 nuevo repetido
ejecutar_pipeline()

d = diferencias(antes, foto_gold()); display(d)
assert contar_id('evt_test_dup', 'bronze') == 1,               'El nuevo duplicado entró más de una vez'
assert all(contar_id(e['event_id'], 'bronze') == 1 for e in reenviados), 'Un reenvío se duplicó'
assert len(d) == 1 and d.eventos_despues.iloc[0] - d.eventos_antes.iloc[0] == 1, 'Gold cambió de más'
RESULTADOS.append({'prueba': '1 · Duplicados (3 reenviados + 1 nuevo repetido)', 'resultado': 'OK',
                   'detalle': 'Gold cambió en 1 fila (+1 evento, +1,00 USD): solo el evento nuevo, una vez'})
print(' Prueba 1 OK')

 lote: {'batch_id': 2, 'entrada': 1, 'validos': 1, 'nuevos': 1, 'fechas': ['2025-07-29'], 'ya_existian': 0}
 lote: {'batch_id': 3, 'entrada': 0, 'validos': 0, 'nuevos': 0, 'fechas': [], 'ya_existian': 0}
 fechas recalculadas: ['2025-07-29']


,org_id,event_date,service,daily_cost_usd_antes,daily_cost_usd_despues,eventos_antes,eventos_despues
2000,org_nam148p0,2025-07-29,networking,0.0357,1.0357,1,2


 Prueba 1 OK


In [9]:
#prueba 2, evento extremadamente tardio
# con un watermark de 2h sobre el tiempo del evento, spark habria descartado este evento: llego 59 dias tarde.
# entra aca, se recalcula solo el 03/07 y "updated_at" prueba que las otras 59 particiones no se tocaron

antes = foto_gold()
base_0307 = next(json.loads(l) for h in HISTORIA for l in open(h)
                 if json.loads(l)['timestamp'].startswith('2025-07-03'))
tardio = {**base_0307, 'event_id': 'evt_test_tardio', 'timestamp': '2025-07-03T10:00:00Z',
          'cost_usd_increment': 12.3456}
escribir_archivo('t2_tardio.jsonl', [tardio])                 # llega último, 59 días tarde
fechas = ejecutar_pipeline()

despues = foto_gold(); d = diferencias(antes, despues); display(d)
delta = (d.daily_cost_usd_despues - d.daily_cost_usd_antes.fillna(0)).iloc[0]
otras = (antes[antes.event_date.astype(str) != '2025-07-03'].sort_values(['org_id', 'event_date', 'service'])
         .reset_index(drop=True))
otras_despues = (despues[despues.event_date.astype(str) != '2025-07-03'].sort_values(['org_id', 'event_date', 'service'])
                 .reset_index(drop=True))
assert fechas == ['2025-07-03'],             'Se recalcularon otras fechas'
assert contar_id('evt_test_tardio', 'bronze') == 1, 'El evento tardío se perdió'
assert len(d) == 1 and abs(delta - 12.3456) < 1e-4, 'El total del 03/07 no se corrigió bien'
assert (otras.updated_at == otras_despues.updated_at).all(), 'Se tocaron particiones de otros días'
RESULTADOS.append({'prueba': '2 · Evento extremadamente tardío (03/07, llega último)', 'resultado': 'OK',
                   'detalle': 'No se descartó; solo se recalculó el 03/07; +12,3456 USD exactos; otros 59 días intactos'})
print(' Prueba 2 OK')

 lote: {'batch_id': 4, 'entrada': 1, 'validos': 1, 'nuevos': 1, 'fechas': ['2025-07-03'], 'ya_existian': 0}
 lote: {'batch_id': 5, 'entrada': 0, 'validos': 0, 'nuevos': 0, 'fechas': [], 'ya_existian': 0}
 fechas recalculadas: ['2025-07-03']


,org_id,event_date,service,daily_cost_usd_antes,daily_cost_usd_despues,eventos_antes,eventos_despues
2083,org_okep7y6w,2025-07-03,storage,2.2682,14.6138,1,2


 Prueba 2 OK


In [10]:
# prueba 3, micro-lote reprocesado despues de un fallo

antes = foto_gold()
base_0815 = next(json.loads(l) for h in HISTORIA for l in open(h)
                 if json.loads(l)['timestamp'].startswith('2025-08-15'))
evento = {**base_0815, 'event_id': 'evt_test_falla', 'cost_usd_increment': 7.0}
escribir_archivo('t3_falla.jsonl', [evento])

print('── Intento 1: falla después de escribir en Bronze')
FALLA['activa'] = True
ejecutar_pipeline()
en_bronze, d1 = contar_id('evt_test_falla', 'bronze'), diferencias(antes, foto_gold())
print(f'   Bronze tiene el evento: {en_bronze} vez · filas cambiadas en Gold: {len(d1)}')

print('── Intento 2: reinicio con el mismo checkpoint')
ejecutar_pipeline()
d2 = diferencias(antes, foto_gold()); display(d2)
assert en_bronze == 1 and len(d1) == 0,              'Estado intermedio inesperado'
assert contar_id('evt_test_falla', 'bronze') == 1,   'El reproceso duplicó en Bronze'
assert contar_id('evt_test_falla', 'silver') == 1,   'El evento no llegó a Silver'
assert len(d2) == 1 and abs((d2.daily_cost_usd_despues - d2.daily_cost_usd_antes).iloc[0] - 7.0) < 1e-4, \
       'Gold no refleja el evento exactamente una vez'
RESULTADOS.append({'prueba': '3 · Micro-lote reprocesado después de un fallo', 'resultado': 'OK',
                   'detalle': 'Tras la falla: en Bronze pero no en Gold; tras reiniciar: ya_existian = 1, '
                              'Silver y Gold actualizados una sola vez (+7,00 USD)'})
print('Prueba 3 OK')

── Intento 1: falla después de escribir en Bronze


ERROR:py4j.clientserver:There was an exception while executing the Python Proxy on the Python Side.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/py4j/clientserver.py", line 644, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/usr/local/lib/python3.13/dist-packages/pyspark/sql/utils.py", line 165, in call
    raise e
  File "/usr/local/lib/python3.13/dist-packages/pyspark/sql/utils.py", line 162, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_521/3827211374.py", line 22, in procesar_lote
    raise RuntimeError(f'Falla simulada en el lote {batch_id}')
RuntimeError: Falla simulada en el lote 6


  stream detenido: [STREAM_FAILED] Query [id = b1c43617-54cf-4c4d-a00c-02ee21e8d20b, runId = c4eb28b7-687f-41bc-a7e6-ccb06f3a0fe0] terminated with exception: [FOREACH_BA
  el job se cayó: Silver y Gold NO se actualizan en esta corrida
   Bronze tiene el evento: 1 vez · filas cambiadas en Gold: 0
── Intento 2: reinicio con el mismo checkpoint
 lote: {'batch_id': 6, 'entrada': 1, 'validos': 1, 'nuevos': 0, 'fechas': ['2025-08-15'], 'ya_existian': 1}
 lote: {'batch_id': 7, 'entrada': 0, 'validos': 0, 'nuevos': 0, 'fechas': [], 'ya_existian': 0}
 fechas recalculadas: ['2025-08-15']


,org_id,event_date,service,daily_cost_usd_antes,daily_cost_usd_despues,eventos_antes,eventos_despues
3039,org_ykl1cq99,2025-08-15,networking,1.1961,8.1961,1,2


Prueba 3 OK


In [11]:
#prueba 4, un duplicado que llega hasta silver
# en caso que un duplicado llegue a silver, el dedupe de gold lo frena, esta es la garantia final que agregamos por el feedback

antes = foto_gold()
fila = spark.read.parquet(str(P['silver'])).filter(F.col('event_id') == 'evt_test_tardio')
fila.write.mode('append').partitionBy('event_date', 'service').parquet(str(P['silver']))   # inyección directa
print('Copias de evt_test_tardio en Silver:', contar_id('evt_test_tardio', 'silver'))

recalcular_gold(['2025-07-03'])
d = diferencias(antes, foto_gold()); display(d)
assert contar_id('evt_test_tardio', 'silver') == 2, 'La inyección no se hizo'
assert len(d) == 0, 'Gold contó el duplicado'
RESULTADOS.append({'prueba': '4 · Duplicado que llega hasta Silver (barreras anteriores burladas)', 'resultado': 'OK',
                   'detalle': 'Silver tiene 2 copias; Gold no cambia gracias al dedupe por event_id antes de agregar'})
print('Prueba 4 OK')

Copias de evt_test_tardio en Silver: 2


,org_id,event_date,service,daily_cost_usd_antes,daily_cost_usd_despues,eventos_antes,eventos_despues


Prueba 4 OK


In [12]:
# resumen y evidencia

resumen = pd.DataFrame(RESULTADOS)
display(resumen)
with open(RUTAS['logs'] / 'pruebas_controladas.json', 'w') as f:
    json.dump({'resultados': RESULTADOS, 'lotes': LOG}, f, indent=2, default=str)
guardar_en_drive('pruebas', 'logs')

,prueba,resultado,detalle
0,1 · Duplicados (3 reenviados + 1 nuevo repetido),OK,"Gold cambió en 1 fila (+1 evento, +1,00 USD): ..."
1,"2 · Evento extremadamente tardío (03/07, llega...",OK,No se descartó; solo se recalculó el 03/07; +1...
2,3 · Micro-lote reprocesado después de un fallo,OK,Tras la falla: en Bronze pero no en Gold; tras...
3,4 · Duplicado que llega hasta Silver (barreras...,OK,Silver tiene 2 copias; Gold no cambia gracias ...


✓ sincronizado: pruebas
✓ sincronizado: logs


## Cierre: pruebas controladas

**donde estasmo:** este notebook prueba el pipeline completo (Streaming -> bronze -> silver -> gold) en un entorno aislados (`pruebas/`). Usa las mismas reglas, umbrales y dimensiones que los notebooks 02-04, sin tocar el lake real.

**linea base:** 10 archivos de historia, 3600 eventos, gold con 3118 filas y 13.345,52 USD en las 60 fechas.

| # | Caso | Qué pasó | Qué barrera actuó |
|---|------|----------|-------------------|
| 1 | **Duplicados**: 3 eventos reenviados + 1 evento nuevo escrito 2 veces | Al lote llegó 1 sola fila. En Gold cambió 1 fila (+1 evento, +1,00 USD) | Barrera 2: `dropDuplicatesWithinWatermark` (los duplicados llegaron dentro de la ventana de 1 h) |
| 2 | **Evento extremadamente tardío**: `timestamp` del 03/07, llega último (59 días tarde) | No se descartó. Solo se recalculó el 03/07 (+12,3456 USD exactos). El `updated_at` de las otras 59 fechas quedó igual | El watermark es sobre `ingest_ts` (llegada), no sobre el tiempo del evento. Gold se recalcula **por fecha afectada** |
| 3 | **Micro-lote reprocesado después de un fallo**: falla simulada después de escribir en Bronze | Intento 1: el evento quedó en Bronze, pero Silver y Gold no cambiaron. Intento 2 (mismo checkpoint): `ya_existian: 1`, se recalculó el 15/08 y Gold sumó +7,00 USD **una sola vez** | Checkpoint (el lote se repite) + barrera 3: anti-join contra Bronze |
| 4 | **Duplicado que llega hasta Silver** (inyectado a mano, saltando las barreras anteriores) | Silver tiene 2 copias. Gold no cambió | Barrera final: `dropDuplicates(['event_id'])` en Gold antes de agregar |

**Conclusiones**
- **Exactly-once efectivo en Gold:** `foreachBatch` es *at-least-once*, pero el anti-join, la sobrescritura por partición y el dedupe final hacen que repetir un lote no cambie el resultado.
- **Defensa en capas:** cada prueba fue frenada por una barrera distinta. Esto responde directamente al feedback de la Entrega 1 sobre deduplicación.
- **Datos tardíos sin pérdida:** Gold no depende del watermark para ser correcto. Recalcula las fechas completas desde Silver y sobrescribe solo esas particiones.

**Corrección encontrada en esta etapa** (va al plan de correcciones): las fechas a recalcular se toman de **todos los eventos válidos del lote**, no solo de los nuevos. Sin este cambio, en la prueba 3 el reintento habría dado `nuevos = 0`, no se habría recalculado nada y el evento habría quedado en Bronze sin llegar nunca a Gold.

**Evidencia:** `logs/pruebas_controladas.json` (resultados + log de cada lote), sincronizado a Drive junto con `pruebas/`.